# Module 02 — Multi-turn by hand

**THE ONE IDEA:** the API is stateless. It stores nothing between calls.
**Memory *is* the list you carry.**

Module 01 sent one message and read one reply. Every agent from module 07 onward is a
loop that appends to a list. This notebook builds that list by hand so the loop later
has no magic left in it.

Runs on `gpt-4.1-mini` — cheap, and with no thinking tokens the per-turn numbers stay clean.

In [1]:
# ── Cell 1: setup ───────────────────────────────────────────────────────────
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py live in the phase root
from _providers import get_client

client, MODEL, _ = get_client("openai")
MAX_TOK = 300

def ask(messages):
    """One API call. Returns (text, input_tokens, output_tokens).
    Note what is NOT here: no session id, no conversation id, no state."""
    r = client.chat.completions.create(
        model=MODEL, max_tokens=MAX_TOK, messages=messages,
    )
    return (r.choices[0].message.content.strip(),
            r.usage.prompt_tokens, r.usage.completion_tokens)

print("client ready:", MODEL)

client ready: gpt-4.1-mini


## Part 1 — Prove the server remembers nothing

Two calls, two independent lists. If the server kept state, the second call would know
about the first. Watch it fail.

In [2]:
# ── Cell 2: two independent calls ───────────────────────────────────────────
t1, _, _ = ask([{"role": "user",
                 "content": "My name is Sameer and I want a 5-year fix."}])
print("CALL 1 ->", t1[:110])

# A brand-new list. Nothing links it to the call above.
t2, _, _ = ask([{"role": "user",
                 "content": "What is my name and what did I ask for?"}])
print("\nCALL 2 ->", t2[:200])

print("\n^ It has no idea. No session id was ever sent, because there isn't one.")

CALL 1 -> Hi Sameer! Could you please clarify what kind of "5-year fix" you're referring to? Are you looking for a finan

CALL 2 -> I’m not able to know your name or personal details unless you share them with me. Also, you haven't asked a question yet. How can I assist you today?

^ It has no idea. No session id was ever sent, because there isn't one.


## Part 2 — Carry the list yourself

Same question as Call 2. The only change: we append each turn to a list we own,
and re-send the whole thing. That is the entire mechanism behind 'memory'.

In [3]:
# ── Cell 3: the hand-carried conversation ───────────────────────────────────
messages = []          # <- THIS is the memory. Nothing else.
rows = []

def turn(user_text):
    messages.append({"role": "user", "content": user_text})
    text, tin, tout = ask(messages)             # re-send EVERYTHING
    messages.append({"role": "assistant", "content": text})
    rows.append((len(messages), user_text, tin, tout))
    return text

turn("My name is Sameer and I want a 5-year fix.")
print("TURN 2 ->", turn("What is my name and what did I ask for?")[:200])
print("\nTURN 3 ->", turn("Summarise everything we have discussed.")[:200])

print("\n^ Same question as before, now answered. The model did not gain memory.")
print("  We gained a list.")

TURN 2 -> Your name is Sameer, and you asked for a "5-year fix."

TURN 3 -> Sure! You introduced yourself as Sameer and mentioned that you want a "5-year fix." I asked for clarification about what kind of 5-year fix you were referring to, such as a fixed-rate mortgage or anot

^ Same question as before, now answered. The model did not gain memory.
  We gained a list.


## Part 3 — What that list costs

Re-sending everything is not free. Input tokens climb every turn while the
answer stays the same size.

In [4]:
# ── Cell 4: the price of memory ─────────────────────────────────────────────
print(f"{'turn':>4} {'msgs':>5} {'in':>6} {'out':>5}   user said")
print("-" * 74)
for i, (n, u, tin, tout) in enumerate(rows, 1):
    print(f"{i:4} {n:5} {tin:6} {tout:5}   {u[:40]}")

first_in, last_in = rows[0][2], rows[-1][2]
print("-" * 74)
print(f"input tokens {first_in} -> {last_in}  ({last_in / first_in:.1f}x over {len(rows)} turns)")
print()
print("LESSON — the API is stateless. Memory IS the list, and you own it.")
print("You re-send the whole conversation every single turn, so INPUT grows with")
print("conversation length while OUTPUT does not. A 10-step agent pays this on every")
print("step — most of why an agent costs ~10x a single call. Module 11 prices it.")
print("Module 22 is where you start throwing parts of this list away.")

turn  msgs     in   out   user said
--------------------------------------------------------------------------
   1     2     21    51   My name is Sameer and I want a 5-year fi
   2     4     91    16   What is my name and what did I ask for?
   3     6    123    66   Summarise everything we have discussed.
--------------------------------------------------------------------------
input tokens 21 -> 123  (5.9x over 3 turns)

LESSON — the API is stateless. Memory IS the list, and you own it.
You re-send the whole conversation every single turn, so INPUT grows while
OUTPUT stays flat. A 10-step agent pays this on every step — that is most of
why an agent costs ~10x a single call. Module 11 puts a dollar figure on it.
Module 22 is where you start throwing parts of this list away.


---

**Next:** `03_structured_output.ipynb` — why parsing free text is a bug, and how
constrained output deletes a whole failure class.